# Deepgram으로 오디오 파일을 전사하고 Anthropic으로 인터뷰 질문 만들기!

**이 노트북을 여러분의 드라이브에 복사한 뒤 아래 안내를 따르세요!** 🥳🥳🥳

----------------------------

# 시작하기
아래 세 셀을 실행하면 원하는 오디오를 전사할 수 있습니다. 아래 주석에서 출력을 원하는 대로 바꾸기 위해 조정할 수 있는 변수를 안내합니다.

이 노트북을 실행하기 전에 전사할 오디오 URL이 몇 개 필요합니다. 원하는 오디오 파일이면 무엇이든 괜찮습니다.

그리고 아직 Deepgram에 가입하지 않으셨다면 여기를 확인해 보세요: https://dpgr.am/prerecorded-notebook-signup

# 1단계: 의존성

이 셀을 실행해 필요한 의존성을 모두 내려받으세요.

참고: 셀은 왼쪽 재생 버튼을 누르거나, 셀을 클릭한 뒤 `shift`+`ENTER`를 동시에 눌러 실행할 수 있습니다. (Mac에서는 `shift` + `return`)

In [ ]:
! pip install requests ffmpeg-python
! pip install deepgram-sdk --upgrade
! pip install requests
! pip install anthropic

# 2단계: 오디오 URL 파일

이 노트북에서 사용할 수 있도록 서버에 올려 둔 오디오 파일을 준비하세요. 또는 아래 코드에 Deepgram이 제공하는 예시 파일이 있습니다.

In [ ]:
# Have you completed Step 2 above? 👀
# Do you see your audio file in the folder on the left? 📂

# 3단계: 전사

다음 변수를 채우세요.


* `DG_KEY` = 여러분의 Deepgram API 키
* `AUDIO_FILE_URL` = 전사하려는 오디오 파일의 URL


이제 셀을 실행하세요! (`Shift` + `Enter`)

-----------



그리고 이미 Deepgram 사용자인데 이 셀에서 오류가 난다면, 가장 흔한 해결 방법은 다음과 같습니다.

1. deepgram-sdk 설치를 업데이트해야 할 수 있습니다.
2. Deepgram 계정에 크레딧이 얼마나 남았는지 확인해야 할 수 있습니다.

In [ ]:
import requests
from deepgram import DeepgramClient, FileSource, PrerecordedOptions

# Deepgram API key
DG_KEY = "🔑🔑🔑 Your API Key here! 🔑🔑🔑"

# URL of the audio file
AUDIO_FILE_URL = "https://static.deepgram.com/examples/nasa-spacewalk-interview.wav"

# Path to save the transcript JSON file
TRANSCRIPT_FILE = "transcript.json"


def main():
    try:
        # STEP 1: Create a Deepgram client using the API key
        deepgram = DeepgramClient(DG_KEY)

        # Download the audio file from the URL
        response = requests.get(AUDIO_FILE_URL, timeout=60)
        if response.status_code == 200:
            buffer_data = response.content
        else:
            print("Failed to download audio file")
            return

        payload: FileSource = {
            "buffer": buffer_data,
        }

        # STEP 2: Configure Deepgram options for audio analysis
        options = PrerecordedOptions(
            model="nova-2",
            smart_format=True,
        )

        # STEP 3: Call the transcribe_file method with the text payload and options
        response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)

        # STEP 4: Write the response JSON to a file
        with open(TRANSCRIPT_FILE, "w") as transcript_file:
            transcript_file.write(response.to_json(indent=4))

        print("Transcript JSON file generated successfully.")

    except Exception as e:
        print(f"Exception: {e}")


if __name__ == "__main__":
    main()

위 셀이 성공하면 content 디렉터리에 JSON 출력 파일이 보일 것입니다. 참고: 셀 실행이 끝난 뒤 JSON 파일이 실제로 나타나기까지 약간의 지연이 있을 수 있습니다. 정상입니다. 파일이 나타날 때까지 잠시 기다리세요.

# 4단계: 전사 결과 확인하기

아래 함수는 출력 JSON을 파싱해 방금 전사한 파일 중 하나의 전사 결과를 출력합니다. (확인하려는 파일이 content 디렉터리에 이미 올라와 있는지 확인하세요.)

**`OUTPUT` 변수에 전사 결과를 보고 싶은 파일 이름을 지정하세요.**

그런 다음 이 셀을 실행하면(`Shift`+`Enter`) 오디오의 문장 단위 전사 결과를 볼 수 있습니다!

In [ ]:
import json

# Set this variable to the path of the output file you wish to read
OUTPUT = "transcript.json"


# The JSON is loaded with information, but if you just want to read the
# transcript, run the code below!
def print_transcript(transcription_file):
    with open(transcription_file) as file:
        data = json.load(file)
        result = data["results"]["channels"][0]["alternatives"][0]["transcript"]
        result = result.split(".")
        for sentence in result:
            print(sentence + ".")


print_transcript(OUTPUT)


위 셀이 성공하면 오디오 전사의 일반 텍스트 버전을 볼 수 있습니다.

# 5단계: Anthropic으로 인터뷰 질문 준비하기

이제 전사본을 Anthropic에 보내 분석하고 인터뷰 질문을 준비해 볼 수 있습니다. 아래 셀을 실행하면(`Shift`+`Enter`) 위 오디오 전사본을 바탕으로 Anthropic이 제안하는 인터뷰 질문 목록을 받을 수 있습니다.

In [ ]:
import json

import anthropic

transcription_file = "transcript.json"


# Function to get the transcript from the JSON file
def get_transcript(transcription_file):
    with open(transcription_file) as file:
        data = json.load(file)
        result = data["results"]["channels"][0]["alternatives"][0]["transcript"]
        return result


# Load the transcript from the JSON file
message_text = get_transcript(transcription_file)

# Initialize the Claude API client
client = anthropic.Anthropic(
    # Defaults to os.environ.get("ANTHROPIC_API_KEY")
    # Claude API key
    api_key="🔑🔑🔑 Your API Key here! 🔑🔑🔑"
)

# Prepare the text for the API request
formatted_messages = [{"role": "user", "content": message_text}]

# Generate thoughtful, open-ended interview questions
response = client.messages.create(
    model="claude-opus-4-1",
    max_tokens=1000,
    temperature=0.5,
    system="Your task is to generate a series of thoughtful, open-ended questions for an interview based on the given context. The questions should be designed to elicit insightful and detailed responses from the interviewee, allowing them to showcase their knowledge, experience, and critical thinking skills. Avoid yes/no questions or those with obvious answers. Instead, focus on questions that encourage reflection, self-assessment, and the sharing of specific examples or anecdotes.",
    messages=formatted_messages,
)

# Print the generated questions

# Join the text of each TextBlock into a single string
content = "".join(block.text for block in response.content)

# Split the content by '\n\n'
parts = content.split("\n\n")

# Print each part with an additional line break
for part in parts:
    print(part)
    print("\n")

이 셀이 성공했다면 원본 오디오 파일을 바탕으로 한 인터뷰 질문 목록이 보일 것입니다. 이제 Deepgram으로 오디오를 전사하고 Anthropic으로 인터뷰 질문 목록을 얻을 수 있습니다.